In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from pymongo import MongoClient
from datetime import datetime
import time


# ===== CONFIGURACIÓN =====
NOMBRE_GRUPO = "E-Commerce_iPhones"
URL_OBJETIVO = "https://listado.mercadolibre.cl/iphone"

# ===== MONGODB LOCAL =====
client = MongoClient('mongodb://bigdata_mongodb:27017/')
db = client['proyecto_bigdata']
coleccion = db['smartphones_ml']

print("=" * 70)
print("SCRAPING CON INTERFAZ VISUAL HABILITADA")
print("=" * 70)

# ===== CONFIGURACIÓN DE SELENIUM (MODO VISIBLE) =====
options = Options()

# ¡IMPORTANTE! NO usar --headless para poder ver el navegador
# options.add_argument("--headless")  # ← COMENTADO para modo visual

# Argumentos de estabilidad para Docker
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")

# User-Agent para parecer navegador real
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

print("\n Iniciando navegador Chrome...")
print(" Ve a http://localhost:6080/vnc.html para ver el navegador\n")

# Inicializar driver
driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

try:
    # ===== FASE 1: ABRIR MERCADO LIBRE =====
    print(f"🔗 Abriendo: {URL_OBJETIVO}")
    driver.get(URL_OBJETIVO)
    
    # ===== PAUSA PARA INTERVENCIÓN HUMANA =====
    print("\n" + "=" * 70)
    print(" INTERVENCIÓN HUMANA REQUERIDA")
    print("=" * 70)
    print("\n INSTRUCCIONES:")
    print("1. Ve a la pestaña del escritorio del contenedor (VNC)")
    print("   URL: http://localhost:6080/vnc.html")
    print("\n2. Verifica si aparece:")
    print("   ✓ Banner de cookies → Aceptar")
    print("   ✓ CAPTCHA → Resolver manualmente")
    print("   ✓ Cualquier popup → Cerrar")
    print("\n3. Una vez que veas el LISTADO DE iPHONES, regresa aquí")
    print("\n4. Presiona ENTER en esta celda para continuar...")
    print("=" * 70)
    
    input("\n Esperando tu confirmación (presiona ENTER cuando estés listo)...\n")
    
    # ===== FASE 2: VERIFICACIÓN DE BLOQUEO =====
    print(" Verificando estado de la página...")
    
    html_check = driver.page_source.lower()
    
    if "captcha" in html_check or "robot" in html_check or "unusual" in html_check:
        print("  ADVERTENCIA: Aún se detecta posible bloqueo en el HTML")
        print("   Palabras encontradas: 'captcha', 'robot' o 'unusual activity'")
        print("\n Esto puede ser un falso positivo (comentarios en el código)")
        print("   Verificando título de la página...")
        
        # Verificación más confiable con el título
        if "robot" in driver.title.lower() or "captcha" in driver.title.lower():
            print(" BLOQUEO REAL DETECTADO en el título de la página")
            print("   Por favor, resuelve el CAPTCHA en la ventana VNC")
            input("   Presiona ENTER cuando lo hayas resuelto...")
        else:
            print(" Título verificado: No hay bloqueo real")
    else:
        print(" Sin bloqueos evidentes detectados")
    
    # ===== FASE 3: CAPTURA DE DATOS =====
    print("\n Iniciando extracción de datos...")
    
    # Esperar que cargue completamente
    time.sleep(3)
    
    # Buscar productos
    items = driver.find_elements(By.CSS_SELECTOR, "li.ui-search-layout__item")
    print(f" Productos encontrados: {len(items)}")
    
    if len(items) == 0:
        print("\n ERROR: No se encontraron productos")
        print("   Posibles causas:")
        print("   1. El selector CSS cambió")
        print("   2. La página no cargó completamente")
        print("   3. Mercado Libre bloqueó el acceso")
        print("\n   Revisa la ventana VNC para ver qué está mostrando el navegador")
    else:
        # Procesar solo los primeros 5 como prueba
        print(f"\n Procesando primeros 5 productos como prueba...\n")
        
        contador = 0
        for i, item in enumerate(items[:5], 1):
            try:
                # Extraer título
                titulo = item.find_element(By.CSS_SELECTOR, ".poly-component__title").text
                
                # Extraer precio actual
                precio_actual_text = item.find_element(By.CSS_SELECTOR, ".andes-money-amount__fraction").text
                precio_actual = int(precio_actual_text.replace(".", ""))
                
                # Intentar extraer precio original (si hay descuento)
                try:
                    precio_original_text = item.find_element(By.CSS_SELECTOR, ".andes-money-amount--previous .andes-money-amount__fraction").text
                    precio_original = int(precio_original_text.replace(".", ""))
                    tiene_descuento = True
                    porcentaje_desc = round(((precio_original - precio_actual) / precio_original) * 100, 2)
                except:
                    precio_original = precio_actual
                    tiene_descuento = False
                    porcentaje_desc = 0.0
                
                # Crear documento
                documento = {
                    "titulo": titulo,
                    "precio_actual": precio_actual,
                    "precio_original": precio_original,
                    "descuento": tiene_descuento,
                    "porcentaje_descuento": porcentaje_desc,
                    "pagina": 1,
                    "grupo": NOMBRE_GRUPO,
                    "timestamp": datetime.now(),
                    "metodo_captura": "visual_intervencion_humana"
                }
                
                # Insertar en MongoDB
                coleccion.insert_one(documento)
                contador += 1
                
                # Mostrar progreso
                print(f" {i}. {titulo[:50]}...")
                print(f"    ${precio_actual:,} CLP", end="")
                if tiene_descuento:
                    print(f" (antes: ${precio_original:,} - {porcentaje_desc}% OFF)")
                else:
                    print()
                print()
                
            except Exception as e:
                print(f"  {i}. Error en este producto: {e}\n")
                continue
        
        print("=" * 70)
        print(f" CAPTURA COMPLETADA")
        print(f" Productos guardados: {contador}")
        print(f" Total en MongoDB: {coleccion.count_documents({'grupo': NOMBRE_GRUPO})}")
        print("=" * 70)

except Exception as e:
    print(f"\n ERROR CRÍTICO: {e}")
    print("\n Soluciones:")
    print("1. Verifica que Docker esté corriendo")
    print("2. Revisa la ventana VNC (localhost:6080)")
    print("3. Asegúrate de que el selector CSS sea correcto")

finally:
    print("\n Cerrando navegador...")
    driver.quit()
    print(" Navegador cerrado correctamente")

SCRAPING CON INTERFAZ VISUAL HABILITADA

 Iniciando navegador Chrome...
 Ve a http://localhost:6080/vnc.html para ver el navegador

🔗 Abriendo: https://listado.mercadolibre.cl/iphone

 INTERVENCIÓN HUMANA REQUERIDA

 INSTRUCCIONES:
1. Ve a la pestaña del escritorio del contenedor (VNC)
   URL: http://localhost:6080/vnc.html

2. Verifica si aparece:
   ✓ Banner de cookies → Aceptar
   ✓ CAPTCHA → Resolver manualmente
   ✓ Cualquier popup → Cerrar

3. Una vez que veas el LISTADO DE iPHONES, regresa aquí

4. Presiona ENTER en esta celda para continuar...



 Esperando tu confirmación (presiona ENTER cuando estés listo)...
 


 Verificando estado de la página...
  ADVERTENCIA: Aún se detecta posible bloqueo en el HTML
   Palabras encontradas: 'captcha', 'robot' o 'unusual activity'

 Esto puede ser un falso positivo (comentarios en el código)
   Verificando título de la página...
 Título verificado: No hay bloqueo real

 Iniciando extracción de datos...
 Productos encontrados: 52

 Procesando primeros 5 productos como prueba...

 1. Apple iPhone 16e (128 Gb) - Blanco - Distribuidor ...
    $699,990 CLP (antes: $699,990 - 0.0% OFF)

 2. Apple iPhone 17 (256 GB) - Azul neblina - Distribu...
    $999,990 CLP (antes: $999,990 - 0.0% OFF)

 3. Apple iPhone 17 Pro Max (256 GB) - Naranja cósmico...
    $1,486,009 CLP

 4. Apple iPhone 15 (128 GB) - Azul - Distribuidor Aut...
    $899,990 CLP (antes: $899,990 - 0.0% OFF)

 5. Celular Fossibot F109s Dual Sim 256 GB ROM Azul 24...
    $299,990 CLP (antes: $299,990 - 0.0% OFF)

 CAPTURA COMPLETADA
 Productos guardados: 5
 Total en MongoDB: 5

 Cerrando navegador...
 Na